# LATIF SILMA Audiobook Lab
This notebook uses the open SILMA TTS v1 model. It does not reproduce the hosted Manus Charon voice. Use only an authorized reference voice. Output is raw concatenated WAV: no fades, normalization, music, or effects.

In [ ]:
!pip -q install silma-tts soundfile numpy
!apt-get -qq update && apt-get -qq install -y ffmpeg

In [ ]:
from google.colab import files
uploaded = files.upload()
print(list(uploaded))

In [ ]:
from pathlib import Path
import re, wave, json, hashlib
from silma_tts.api import SilmaTTS

TEXT_FILE = Path('/content/manuscript.txt')
REFERENCE_AUDIO = Path('/content/reference.wav')
REFERENCE_TEXT = 'ويدقق النظر في القرآن الكريم وسائر الكتب السماوية ويتبع مسالك الرسل العظام عليهم الصلاة والسلام.'
OUT = Path('/content/audiobook_output'); OUT.mkdir(exist_ok=True)
MAX_CHARS = 4200
SPEED = 1.0
SEED = None

def split_text(text, max_chars=MAX_CHARS):
    text = re.sub(r'\s+', ' ', text).strip()
    parts=[]
    while text:
        if len(text) <= max_chars: parts.append(text); break
        cut=max(text.rfind('،',0,max_chars), text.rfind(' ',0,max_chars))
        if cut < max_chars//2: cut=max_chars
        parts.append(text[:cut].strip()); text=text[cut:].strip()
    return parts

text = TEXT_FILE.read_text(encoding='utf-8')
parts = split_text(text)
tts = SilmaTTS()
clips=[]
for i, part in enumerate(parts, 1):
    path=OUT/f'clip_{i:03d}.wav'
    print(f'{i}/{len(parts)}: {len(part)} chars')
    tts.infer(ref_file=str(REFERENCE_AUDIO), ref_text=REFERENCE_TEXT, gen_text=part, file_wave=str(path), seed=SEED, speed=SPEED)
    clips.append(path)

master=OUT/'audiobook_master.wav'
with wave.open(str(clips[0]), 'rb') as first:
    params=first.getparams(); frames=[first.readframes(first.getnframes())]
for path in clips[1:]:
    with wave.open(str(path), 'rb') as w:
        frames.append(w.readframes(w.getnframes()))
with wave.open(str(master), 'wb') as w:
    w.setparams(params)
    for pcm in frames: w.writeframes(pcm)
print(master, master.stat().st_size)

In [ ]:
from google.colab import files
files.download('/content/audiobook_output/audiobook_master.wav')